In [1]:
""" Import required libraries
"""
import torch
import scipy.io as sio
import numpy as np
import netCDF4 as nc
import deepxde as dde
import pinnicle as pinn
from matplotlib.path import Path
from tqdm import tqdm
dde.config.set_default_float('float32')

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


Set the default float type to float64
Set the default float type to float32


In [ ]:
""" This is the code for evaluating the neural networks to get the data needed for the figures.
"""

In [ ]:
""" specify data paths
"""
glacier = "Byrd-BMap3GL"
size = "30ma"

model = "MCexactHelm"   # for PENN
# model = "MC"            # for PINN

path = '<specify path>' # path to 'PENNs-for-MC-ice-thickness' folder
modelpath = f'{path}/models/'
datapath = f'{path}/data/'
savepath = f'{path}/saved-data/'

outline = datapath + 'Byrd-BMap3GL-30ma-vertices.exp'

app_mb = datapath + 'apparent_mass_balance_2013-17_mps.mat'
bm3 = datapath + 'BedMachineAntarctica-v3.nc' # BedMachine Antarctica v3                not included in this repository, get from original data source
bmap3 = datapath + 'bedmap3.nc' # BedMap3                                               not included in this repository, get from original data source
meas = datapath + 'antarctic_ice_vel_phase_map_v01.nc' # MEaSUREs velocity              not included in this repository, get from original data source
itslive = datapath + 'ITS_LIVE_velocity_120m_RGI19A_0000_v02.nc' # ITS_LIVE velocity    not included in this repository, get from original data source

outFile = glacier
modelFolder = modelpath + outFile 

In [ ]:
""" create the PINNICLE parameter dictionary
"""
hp = {}
hp['save_path'] = modelFolder

hp["activation"] = "SiLU" 
hp["initializer"] = "Glorot uniform"
hp["num_neurons"] = 50
hp["num_layers"] = 8

hp['shapefile'] = outline

MC = {}
MC_exact = {}
MC_exact_Helm = {}
if model == 'MC':
    hp["equations"] = {"MC":MC}
elif model == "MCexactHelm":
    hp["equations"] = {"MC_exact_Helmholtz":MC_exact_Helm}

hp ['data'] = {}

In [ ]:
""" load the trained neural networks
    five per experiment
"""
""" experiments:
    Number      Date
    1           24-06-26
    2           06-07-26
"""
if model == "MCexactHelm":
    expNo = 1
    date = '24-06-26'
elif model == 'MC':
    expNo = 2
    date = '06-07-26'

experiments = [None]*5
for k in np.arange(5):
    experiments[k] = pinn.PINN(hp)
    experiments[k].load_model(path=f'{modelpath}', 
                        subfolder='pinn', 
                        name=f'Byrd-BMap3GL-30ma-{model}-exp{expNo}-No{k+1}-{date}',
                        epochs=700000,
                        fileformat='pt')

add input transform with tensor([ 375692.6875, -983606.9375], device='mps:0') and tensor([ 647310.2500, -645047.0625], device='mps:0')
add output transform with [-3.1709792e-04 -3.1709792e-04 -1.5854896e-07  1.0000000e+01] and [3.1709792e-04 3.1709792e-04 1.5854896e-07 3.5000000e+03]
Compiling model...
'compile' took 0.000143 s

Compiling model...
'compile' took 0.000232 s

add input transform with tensor([ 375692.6875, -983606.9375], device='mps:0') and tensor([ 647310.2500, -645047.0625], device='mps:0')
add output transform with [-3.1709792e-04 -3.1709792e-04 -1.5854896e-07  1.0000000e+01] and [3.1709792e-04 3.1709792e-04 1.5854896e-07 3.5000000e+03]
Compiling model...
'compile' took 0.000052 s

Compiling model...
'compile' took 0.000170 s

add input transform with tensor([ 375692.6875, -983606.9375], device='mps:0') and tensor([ 647310.2500, -645047.0625], device='mps:0')
add output transform with [-3.1709792e-04 -3.1709792e-04 -1.5854896e-07  1.0000000e+01] and [3.1709792e-04 3.17

Compiling model...
'compile' took 0.000160 s



In [7]:
""" load the required data
"""
ds_vel = nc.Dataset(meas)
ds_bmac = nc.Dataset(bm3)
ds_bmap = nc.Dataset(bmap3)
ds_il = nc.Dataset(itslive)

ds_mb = sio.loadmat(app_mb)

In [34]:
""" create the relevant coordinate arrays and grids 
"""
# load data

x_v = ds_vel['x'][:]
y_v = ds_vel['y'][:]
x_v2 = ds_il['x'][:]
y_v2 = ds_il['y'][:]
x_H1 = ds_bmac['x'][:]
y_H1 = ds_bmac['y'][:]
x_H2 = ds_bmap['x'][:]
y_H2 = ds_bmap['y'][:]
x_mb = ds_mb['x'][:]
y_mb = ds_mb['y'][:]

vertices = np.loadtxt(outline,delimiter=' ',skiprows=5)
minx = np.min(vertices[:,0])
maxx = np.max(vertices[:,0])
miny = np.min(vertices[:,1])
maxy = np.max(vertices[:,1])
minmax = [minx, maxx, miny, maxy]


# create masks

xvmask = np.where(np.logical_and(x_v>(minx),x_v<(maxx)))[0]
yvmask = np.where(np.logical_and(y_v>(miny),y_v<(maxy)))[0]

xv2mask = np.where(np.logical_and(x_v2>(minx),x_v2<(maxx)))[0]
yv2mask = np.where(np.logical_and(y_v2>(miny),y_v2<(maxy)))[0]

xH1mask = np.where(np.logical_and(x_H1>(minx),x_H1<(maxx)))[0]
yH1mask = np.where(np.logical_and(y_H1>(miny),y_H1<(maxy)))[0]

xH2mask = np.where(np.logical_and(x_H2>(minx),x_H2<(maxx)))[0]
yH2mask = np.where(np.logical_and(y_H2>(miny),y_H2<(maxy)))[0]

mbmask = np.where(np.logical_and(
    np.logical_and(x_mb>(minx),x_mb<(maxx)),
    np.logical_and(y_mb>(miny),y_mb<(maxy))
    ))


# grid data

Xv,Yv = np.meshgrid(x_v[xvmask],y_v[yvmask])
Xvf = Xv.flatten()
Yvf = Yv.flatten()
Xvf = Xvf.reshape((Xvf.size,1))
Yvf = Yvf.reshape((Yvf.size,1))
XYv = np.hstack((Xvf,Yvf))

Xv2,Yv2 = np.meshgrid(x_v2[xv2mask],y_v2[yv2mask])
Xv2f = Xv2.flatten()
Yv2f = Yv2.flatten()
Xv2f = Xv2f.reshape((Xv2f.size,1))
Yv2f = Yv2f.reshape((Yv2f.size,1))
XYv2 = np.hstack((Xv2f,Yv2f))

XH1,YH1 = np.meshgrid(x_H1[xH1mask],y_H1[yH1mask])
XH1f = XH1.flatten()
YH1f = YH1.flatten()
XH1f = XH1f.reshape((XH1f.size,1))
YH1f = YH1f.reshape((YH1f.size,1))
XYH1 = np.hstack((XH1f,YH1f))

XH2,YH2 = np.meshgrid(x_H2[xH2mask],y_H2[yH2mask])
XH2f = XH2.flatten()
YH2f = YH2.flatten()
XH2f = XH2f.reshape((XH2f.size,1))
YH2f = YH2f.reshape((YH2f.size,1))
XYH2 = np.hstack((XH2f,YH2f))

Xmb = x_mb[mbmask]
Ymb = y_mb[mbmask]
Xmb = Xmb.reshape((Xmb.size,1))
Ymb = Ymb.reshape((Ymb.size,1))
XYmb = np.hstack((Xmb,Ymb))

In [ ]:
# profile coordinates

path_utig = datapath + 'BedMap3-radar-thk-profiles/bedmap3_raw_data/UTIG_2010_ICECAP_AIR_BM3.csv'
path_nasa = datapath + 'BedMap3-radar-thk-profiles/bedmap2_raw_data/NASA_2011_ICEBRIDGE_AIR_BM2.csv'

ds_utig = pd.read_csv(path_utig,skiprows=18,index_col=['trajectory_id'])
ds_nasa = pd.read_csv(path_nasa,skiprows=18,index_col=['trajectory_id'])

profile_A = 2011121303002
profile_B = 'IR1HI2_2010343_WSB_JKB1a_GL0099a'
profile_C = 'IR2HI2_2010343_WSB_JKB2c_GL0233b'
profile_D = 'IR1HI2_2010343_WSB_JKB1a_GL0367a'

transformer_profiles = Transformer.from_crs(
    "EPSG:4326",   # lon/lat (WGS84)
    "EPSG:3031",   # Antarctic Polar Stereographic
    always_xy=True
)

XA,YA = transformer_profiles.transform(ds_nasa.loc[profile_A]['longitude (degree_east)'], ds_nasa.loc[profile_A]['latitude (degree_north)'])
XB,YB = transformer_profiles.transform(ds_utig.loc[profile_B]['longitude (degree_east)'], ds_utig.loc[profile_B]['latitude (degree_north)'])
XC,YC = transformer_profiles.transform(ds_utig.loc[profile_C]['longitude (degree_east)'], ds_utig.loc[profile_C]['latitude (degree_north)'])
XD,YD = transformer_profiles.transform(ds_utig.loc[profile_D]['longitude (degree_east)'], ds_utig.loc[profile_D]['latitude (degree_north)'])

data = sio.loadmat(f'{homepath}/saved-data/exp1-ice-thickness-BedMap-grid.mat')
x = data['x']
y = data['y']
thk = data['median_thickness']

fig,ax = plt.subplots(1,1)
ax1 = ax.pcolormesh(x,y,thk.reshape((x.shape[0],x.shape[1])))
ax.set_aspect(1)

XYA = np.vstack((XA[10:650],YA[10:650])).transpose()
XYB = np.vstack((XB[15700:17600],YB[15700:17600])).transpose()
XYC = np.vstack((XC[1600:3500],YC[1600:3500])).transpose()
XYD = np.vstack((XD[12000:14700],YD[12000:14700])).transpose()

ax.scatter(XYA[:,0],XYA[:,1],s=0.1)
ax.scatter(XYB[:,0],XYB[:,1],s=0.1)
ax.scatter(XYC[:,0],XYC[:,1],s=0.1)
ax.scatter(XYD[:,0],XYD[:,1],s=0.1)



In [ ]:
thk_A = ds_nasa.loc[profile_A]['land_ice_thickness (m)'][10:650]
bed_A = ds_nasa.loc[profile_A]['bedrock_altitude (m)'][10:650]
thk_B = ds_utig.loc[profile_B]['land_ice_thickness (m)'][15700:17600]
bed_B = ds_utig.loc[profile_B]['bedrock_altitude (m)'][15700:17600]
thk_C = ds_utig.loc[profile_C]['land_ice_thickness (m)'][1600:3500]
bed_C = ds_utig.loc[profile_C]['bedrock_altitude (m)'][1600:3500]
thk_D = ds_utig.loc[profile_D]['land_ice_thickness (m)'][12000:14700]
bed_D = ds_utig.loc[profile_D]['bedrock_altitude (m)'][12000:14700]

for i,thk,bed,xy in zip(np.arange(4),[thk_A,thk_B,thk_C,thk_D],[bed_A,bed_B,bed_C,bed_D],[XYA,XYB,XYC,XYD]):
    data = {"x":np.array(xy[:,0]).reshape((len(bed),1)),"y":np.array(xy[:,1]).reshape((len(bed),1)),"thickness":np.array(thk).reshape((len(bed),1)),"bed":np.array(bed).reshape((len(bed),1))}
    sio.savemat(f'{savepath}profile-p{i+1}-bmap3-ice-thickness-observations.mat',data)

In [35]:
""" define the operators for the network prediction 
"""
experiment = experiments[0]

def OP_mf_mag(i,o):
    return experiment.physics.mf_mag(i,o,None)

def OP_mb_MC(i,o):
    return experiment.physics.smb_MC(i,o,None)

def OP_H(i,o):
    return experiment.physics.get_H(i,o)

def OP_vel_mag_MC(i,o):
    return experiment.physics.vel_mag_MC(i,o,None)

def OP_s(i,o):
    return experiment.physics.get_s(i,o)

def OP_u_MC(i,o):
    return experiment.physics.u_MC(i,o,None)

def OP_v_MC(i,o):
    return experiment.physics.v_MC(i,o,None)

def OP_mf_rot(i,o):
    _, _, _, _, Rx, Ry = experiment.physics.DR_xy(i,o)
    return torch.sqrt(Ry**2 + Rx**2)

def OP_mf_div(i,o):
    Dxmb, Dymb, _, _, _, _ = experiment.physics.DR_xy(i,o)
    return torch.sqrt(Dxmb**2 + Dymb**2)

def OP_res_MC(i,o):
    return experiment.physics.pdes(i,o)[0]

In [ ]:
""" evaluate the network prediction 
"""
# create empty arrays

D = np.zeros((5,XYv.shape[0]))
R = np.zeros((5,XYv.shape[0]))

vel_p = np.zeros((5,XYv.shape[0]))
u_p = np.zeros((5,XYv.shape[0]))
v_p = np.zeros((5,XYv.shape[0]))
mf_p  = np.zeros((5,XYv.shape[0]))
mf_rot = np.zeros((5,XYv.shape[0]))
mf_div = np.zeros((5,XYv.shape[0]))
pde_res = np.zeros((5,XYv.shape[0]))
mb_p = np.zeros((5,XYmb.shape[0]))
# H_p = np.zeros((5,XYH.shape[0]))
H_pfull = np.zeros((5,XYH1.shape[0]))
H_pb = np.zeros((5,XYH1.shape[0]))
H_pb2 = np.zeros((5,XYH2.shape[0]))
H_meas = np.zeros((5,XYv.shape[0]))
H_il = np.zeros((5,XYv2.shape[0]))
H_profile_A = np.zeros((5,XYA.shape[0]))
H_profile_B = np.zeros((5,XYB.shape[0]))
H_profile_C = np.zeros((5,XYC.shape[0]))
H_profile_D = np.zeros((5,XYD.shape[0]))

# define step sizes for the loop

step = 50

XYv_step = int(XYv.shape[0]/step)
XYv_rest = int(XYv.shape[0]) - step*XYv_step

XYv2_step = int(XYv2.shape[0]/step)
XYv2_rest = int(XYv2.shape[0]) - step*XYv2_step

XYmb_step = int(XYmb.shape[0]/step)
XYmb_rest = int(XYmb.shape[0]) - step*XYmb_step

XYH1_step = int(XYH1.shape[0]/step)
XYH1_rest = int(XYH1.shape[0]) - step*XYH1_step

XYH2_step = int(XYH2.shape[0]/step)
XYH2_rest = int(XYH2.shape[0]) - step*XYH2_step

XYA_step = int(XYA.shape[0]/step)
XYA_rest = int(XYA.shape[0]) - step*XYA_step

XYB_step = int(XYB.shape[0]/step)
XYB_rest = int(XYB.shape[0]) - step*XYB_step

XYC_step = int(XYC.shape[0]/step)
XYC_rest = int(XYC.shape[0]) - step*XYC_step

XYD_step = int(XYD.shape[0]/step)
XYD_rest = int(XYD.shape[0]) - step*XYD_step

# evaluate the network
for k in np.arange(5):
    for i in tqdm(np.arange(step)):
        if model == 'MC':
            u_p[k,i*XYv_step:(i+1)*XYv_step] = experiments[k].model.predict(XYv[i*XYv_step:(i+1)*XYv_step])[:,0]
            v_p[k,i*XYv_step:(i+1)*XYv_step] = experiments[k].model.predict(XYv[i*XYv_step:(i+1)*XYv_step])[:,1]
            vel_p[k,i*XYv_step:(i+1)*XYv_step] = np.sqrt(u_p[k,i*XYv_step:(i+1)*XYv_step]**2 + v_p[k,i*XYv_step:(i+1)*XYv_step]**2)
            H_pfull[k,i*XYH1_step:(i+1)*XYH1_step] = experiments[k].model.predict(XYH1[i*XYH1_step:(i+1)*XYH1_step])[:,3]
            H_pb2[k,i*XYH2_step:(i+1)*XYH2_step] = experiments[k].model.predict(XYH2[i*XYH2_step:(i+1)*XYH2_step])[:,3]
            H_meas[k,i*XYv_step:(i+1)*XYv_step] = experiments[k].model.predict(XYv[i*XYv_step:(i+1)*XYv_step])[:,3]
            H_profile_A[k,i*XYA_step:(i+1)*XYA_step] = experiments[k].model.predict(XYA[i*XYA_step:(i+1)*XYA_step])[:,3]
            H_profile_B[k,i*XYB_step:(i+1)*XYB_step] = experiments[k].model.predict(XYB[i*XYB_step:(i+1)*XYB_step])[:,3]
            H_profile_C[k,i*XYC_step:(i+1)*XYC_step] = experiments[k].model.predict(XYC[i*XYC_step:(i+1)*XYC_step])[:,3]
            H_profile_D[k,i*XYD_step:(i+1)*XYD_step] = experiments[k].model.predict(XYD[i*XYD_step:(i+1)*XYD_step])[:,3]
            mb_p[k,i*XYmb_step:(i+1)*XYmb_step] = experiments[k].model.predict(XYmb[i*XYmb_step:(i+1)*XYmb_step])[:,2]
            pde_res[k,i*XYv_step:(i+1)*XYv_step] = experiments[k].model.predict(XYv[i*XYv_step:(i+1)*XYv_step],operator=OP_res_MC)[:,0]

        else:
            D[k,i*XYv_step:(i+1)*XYv_step] = experiments[k].model.predict(XYv[i*XYv_step:(i+1)*XYv_step])[:,0]
            R[k,i*XYv_step:(i+1)*XYv_step] = experiments[k].model.predict(XYv[i*XYv_step:(i+1)*XYv_step])[:,1]

            vel_p[k,i*XYv_step:(i+1)*XYv_step] = experiments[k].model.predict(XYv[i*XYv_step:(i+1)*XYv_step],operator=OP_vel_mag_MC)[:,0]

            mf_p[k,i*XYv_step:(i+1)*XYv_step] = experiments[k].model.predict(XYv[i*XYv_step:(i+1)*XYv_step],operator=OP_mf_mag)[:,0]

            mf_rot[k,i*XYv_step:(i+1)*XYv_step] = experiments[k].model.predict(XYv[i*XYv_step:(i+1)*XYv_step],operator=OP_mf_rot)[:,0]
            mf_div[k,i*XYv_step:(i+1)*XYv_step] = experiments[k].model.predict(XYv[i*XYv_step:(i+1)*XYv_step],operator=OP_mf_div)[:,0]
            
            mb_p[k,i*XYmb_step:(i+1)*XYmb_step] = experiments[k].model.predict(XYmb[i*XYmb_step:(i+1)*XYmb_step],operator=OP_mb_MC)[:,0]

            H_pfull[k,i*XYH1_step:(i+1)*XYH1_step] = experiments[k].model.predict(XYH1[i*XYH1_step:(i+1)*XYH1_step],operator=OP_H)[:,0]

            H_profile_A[k,i*XYA_step:(i+1)*XYA_step] = experiments[k].model.predict(XYA[i*XYA_step:(i+1)*XYA_step],operator=OP_H)[:,0]
            H_profile_B[k,i*XYB_step:(i+1)*XYB_step] = experiments[k].model.predict(XYB[i*XYB_step:(i+1)*XYB_step],operator=OP_H)[:,0]
            H_profile_C[k,i*XYC_step:(i+1)*XYC_step] = experiments[k].model.predict(XYC[i*XYC_step:(i+1)*XYC_step],operator=OP_H)[:,0]
            H_profile_D[k,i*XYD_step:(i+1)*XYD_step] = experiments[k].model.predict(XYD[i*XYD_step:(i+1)*XYD_step],operator=OP_H)[:,0]

            H_pb2[k,i*XYH2_step:(i+1)*XYH2_step] = experiments[k].model.predict(XYH2[i*XYH2_step:(i+1)*XYH2_step],operator=OP_H)[:,0]
            H_il[k,i*XYv2_step:(i+1)*XYv2_step] = experiments[k].model.predict(XYv2[i*XYv2_step:(i+1)*XYv2_step],operator=OP_H)[:,0]


    if not XYv_rest==0:
        if model == 'MC':
            pde_res[k,-XYv_rest:] = experiments[k].model.predict(XYv[-XYv_rest:],operator=OP_res_MC)[:,0]
            u_p[k,-XYv_rest:] = experiments[k].model.predict(XYv[-XYv_rest:])[:,0]
            v_p[k,-XYv_rest:] = experiments[k].model.predict(XYv[-XYv_rest:])[:,1]
            H_meas[k,-XYv_rest:] = experiments[k].model.predict(XYv[-XYv_rest:])[:,3]
            vel_p[k,-XYv_rest:] = np.sqrt(u_p[k,-XYv_rest:]**2 + v_p[k,-XYv_rest:]**2)
        else:
            D[k,-XYv_rest:] = experiments[k].model.predict(XYv[-XYv_rest:])[:,0]
            R[k,-XYv_rest:] = experiments[k].model.predict(XYv[-XYv_rest:])[:,1]

            vel_p[k,-XYv_rest:] = experiments[k].model.predict(XYv[-XYv_rest:],operator=OP_vel_mag_MC)[:,0]

            mf_p[k,-XYv_rest:] = experiments[k].model.predict(XYv[-XYv_rest:],operator=OP_mf_mag)[:,0]

            mf_rot[k,-XYv_rest:] = experiments[k].model.predict(XYv[-XYv_rest:],operator=OP_mf_rot)[:,0]
            mf_div[k,-XYv_rest:] = experiments[k].model.predict(XYv[-XYv_rest:],operator=OP_mf_div)[:,0]
        
        
    if not XYmb_rest==0:
        if model=='MC':
            mb_p[k,-XYmb_rest:] = experiments[k].model.predict(XYmb[-XYmb_rest:])[:,2]
        else:
            mb_p[k,-XYmb_rest:] = experiments[k].model.predict(XYmb[-XYmb_rest:],operator=OP_mb_MC)[:,0]

    if not XYH1_rest==0:
        if model=='MC':
            H_pfull[k,-XYH1_rest:] = experiments[k].model.predict(XYH1[-XYH1_rest:])[:,3]
        else:
            H_pfull[k,-XYH1_rest:] = experiments[k].model.predict(XYH1[-XYH1_rest:],operator=OP_H)[:,0]
            
    if not XYH2_rest==0:
        if model=='MC':
            H_pb2[k,-XYH2_rest:] = experiments[k].model.predict(XYH2[-XYH2_rest:])[:,3]
        else:
            H_pb2[k,-XYH2_rest:] = experiments[k].model.predict(XYH2[-XYH2_rest:],operator=OP_H)[:,0]

    if not XYv2_rest==0:
        if model=='MC':
            H_il[k,-XYv2_rest:] = experiments[k].model.predict(XYv2[-XYv2_rest:])[:,3]
        else:
            H_il[k,-XYv2_rest:] = experiments[k].model.predict(XYv2[-XYv2_rest:],operator=OP_H)[:,0]

    if not XYA_rest==0:
        if model=='MC':
            H_profile_A[k,-XYA_rest:] = experiments[k].model.predict(XYA[-XYA_rest:])[:,3]
        else:
            H_profile_A[k,-XYA_rest:] = experiments[k].model.predict(XYA[-XYA_rest:],operator=OP_H)[:,0]

    if not XYB_rest==0:
        if model=='MC':
            H_profile_B[k,-XYB_rest:] = experiments[k].model.predict(XYB[-XYB_rest:])[:,3]
        else:
            H_profile_B[k,-XYB_rest:] = experiments[k].model.predict(XYB[-XYB_rest:],operator=OP_H)[:,0]

    if not XYC_rest==0:
        if model=='MC':
            H_profile_C[k,-XYC_rest:] = experiments[k].model.predict(XYC[-XYC_rest:])[:,3]
        else:
            H_profile_C[k,-XYC_rest:] = experiments[k].model.predict(XYC[-XYC_rest:],operator=OP_H)[:,0]

    if not XYD_rest==0:
        if model=='MC':
            H_profile_D[k,-XYD_rest:] = experiments[k].model.predict(XYD[-XYD_rest:])[:,3]
        else:
            H_profile_D[k,-XYD_rest:] = experiments[k].model.predict(XYD[-XYD_rest:],operator=OP_H)[:,0]

 22%|██▏       | 11/50 [00:00<00:00, 51.52it/s]

100%|██████████| 50/50 [00:00<00:00, 96.04it/s]


In [ ]:
## create masks
poly_path = Path(vertices)
inside_v = poly_path.contains_points(XYv)
inside_H1 = poly_path.contains_points(XYH1)
inside_H2 = poly_path.contains_points(XYH2)
inside_Hmeas = poly_path.contains_points(XYv)
inside_Hil = poly_path.contains_points(XYv2)
inside_mb = poly_path.contains_points(XYmb)

In [12]:
## D, R, mass flux (Measures grid)

# convert from m/s to m/a
spy = 31536000.  # seconds per year
mf_p_ma = mf_p*spy
mf_div_ma = mf_div*spy
mf_rot_ma = mf_rot*spy

# normalise D,R gauges s.t. min(R)==min(D)==0
Rmean = np.nanmean(np.where(inside_v,R,np.nan),axis=1)
Dmean = np.nanmean(np.where(inside_v,D,np.nan),axis=1)
Rnorm = np.zeros(np.shape(R))
Dnorm = np.zeros(np.shape(R))
for i in np.arange(5):
    Rnorm[i,:] = np.where(inside_v,R[i,:] - Rmean[i],np.nan)
    Dnorm[i,:] = np.where(inside_v,D[i,:] - Dmean[i],np.nan)
    Rnorm[i,:] = Rnorm[i,:] - np.nanmin(Rnorm[i,:])
    Dnorm[i,:] = Dnorm[i,:] - np.nanmin(Dnorm[i,:])

meanD = np.mean(np.where(inside_v,Dnorm,np.nan),axis=0)
meanR = np.mean(np.where(inside_v,Rnorm,np.nan),axis=0)
meanMF = np.mean(np.where(inside_v,mf_p_ma,np.nan),axis=0)
meanMFdiv = np.mean(np.where(inside_v,mf_div_ma,np.nan),axis=0)
meanMFrot = np.mean(np.where(inside_v,mf_rot_ma,np.nan),axis=0)

data = {"x":Xv,"y":Yv,"mean_D":meanD,"mean_R":meanR,"mean_MF":meanMF,"mean_MF_div":meanMFdiv,"mean_MF_rot":meanMFrot}
sio.savemat(f'{savepath}exp{expNo}-potentials-and-mass-flux-Measures-grid.mat',data)

In [16]:
## D, R, mass flux (Measures grid)

# convert from m/s to m/a
spy = 31536000.  # seconds per year
mf_p_ma = mf_p*spy
mf_div_ma = mf_div*spy
mf_rot_ma = mf_rot*spy

Dmean = np.nanmean(np.where(inside_v,D,np.nan))
Rmean = np.nanmean(np.where(inside_v,R,np.nan))
Rnorm = np.where(inside_v,R - Rmean,np.nan)
Dnorm = np.where(inside_v,D - Dmean,np.nan)
Rnorm = Rnorm - np.nanmin(Rnorm)
Dnorm = Dnorm - np.nanmin(Dnorm)

MF = np.where(inside_v,mf_p_ma[i,:],np.nan)
MFdiv = np.where(inside_v,mf_div_ma[i,:],np.nan)
MFrot = np.where(inside_v,mf_rot_ma[i,:],np.nan)

data = {"x":Xv,"y":Yv,"D":Dnorm,"R":Rnorm,"MF":MF,"MF_div":MFdiv,"MF_rot":MFrot}
sio.savemat(f'{savepath}exp{expNo}-No1-potentials-and-mass-flux-Measures-grid.mat',data)

In [ ]:
## velocity (Measures grid)

# convert from m/s to m/a
spy = 31536000.  # seconds per year
vel_p_ma = vel_p*spy

mean = np.mean(np.where(inside_v,vel_p_ma,np.nan),axis=0)
median = np.median(np.where(inside_v,vel_p_ma,np.nan),axis=0)
variance = np.var(np.where(inside_v,vel_p_ma,np.nan),axis=0)
mini = np.min(np.where(inside_v,vel_p_ma,np.nan),axis=0)
maxi = np.max(np.where(inside_v,vel_p_ma,np.nan),axis=0)
range = maxi-mini

data = {"x":Xv,"y":Yv,"mean_velocity":mean,"median_velocity":median,"variance":variance,"min_max_range":range}
sio.savemat(f'{savepath}exp{expNo}-surface-velocity-Measures-grid.mat',data)

In [46]:
## ice thickness (BedMachine grid)
mean = np.mean(np.where(inside_H1,H_pfull,np.nan),axis=0)
median = np.median(np.where(inside_H1,H_pfull,np.nan),axis=0)
variance = np.var(np.where(inside_H1,H_pfull,np.nan),axis=0)
mini = np.min(np.where(inside_H1,H_pfull,np.nan),axis=0)
maxi = np.max(np.where(inside_H1,H_pfull,np.nan),axis=0)
range = maxi-mini

data = {"x":XH1,"y":YH1,"mean_thickness":mean,"median_thickness":median,"variance":variance,"min_max_range":range}
sio.savemat(f'{savepath}exp{expNo}-ice-thickness-BedMachine-grid.mat',data)

In [47]:
## ice thickness (BedMap grid)
mean = np.mean(np.where(inside_H2,H_pb2,np.nan),axis=0)
median = np.median(np.where(inside_H2,H_pb2,np.nan),axis=0)
variance = np.var(np.where(inside_H2,H_pb2,np.nan),axis=0)
mini = np.min(np.where(inside_H2,H_pb2,np.nan),axis=0)
maxi = np.max(np.where(inside_H2,H_pb2,np.nan),axis=0)
range = maxi-mini

data = {"x":XH2,"y":YH2,"mean_thickness":mean,"median_thickness":median,"variance":variance,"min_max_range":range}
sio.savemat(f'{savepath}exp{expNo}-ice-thickness-BedMap-grid.mat',data)

In [ ]:
## ice thickness (Measures grid)
mean = np.mean(np.where(inside_Hmeas,H_meas,np.nan),axis=0)
median = np.median(np.where(inside_Hmeas,H_meas,np.nan),axis=0)
variance = np.var(np.where(inside_Hmeas,H_meas,np.nan),axis=0)
mini = np.min(np.where(inside_Hmeas,H_meas,np.nan),axis=0)
maxi = np.max(np.where(inside_Hmeas,H_meas,np.nan),axis=0)
range = maxi-mini

data = {"x":Xv2,"y":Yv2,"mean_thickness":mean,"median_thickness":median,"variance":variance,"min_max_range":range}
sio.savemat(f'{savepath}exp{expNo}-ice-thickness-Measures-grid.mat',data)

In [22]:
## ice thickness (ItsLive grid)
mean = np.mean(np.where(inside_Hil,H_il,np.nan),axis=0)
median = np.median(np.where(inside_Hil,H_il,np.nan),axis=0)
variance = np.var(np.where(inside_Hil,H_il,np.nan),axis=0)
mini = np.min(np.where(inside_Hil,H_il,np.nan),axis=0)
maxi = np.max(np.where(inside_Hil,H_il,np.nan),axis=0)
range = maxi-mini

data = {"x":Xv2,"y":Yv2,"mean_thickness":mean,"median_thickness":median,"variance":variance,"min_max_range":range}
sio.savemat(f'{savepath}exp{expNo}-ice-thickness-ItsLive-grid.mat',data)

In [38]:
## apparent mass balance (BedMap grid)

# convert from m/s to m/a
spy = 31536000.  # seconds per year
mb_p = mb_p*spy


mean = np.mean(np.where(inside_mb,mb_p,np.nan),axis=0)
median = np.median(np.where(inside_mb,mb_p,np.nan),axis=0)
variance = np.var(np.where(inside_mb,mb_p,np.nan),axis=0)
mini = np.min(np.where(inside_mb,mb_p,np.nan),axis=0)
maxi = np.max(np.where(inside_mb,mb_p,np.nan),axis=0)
range = maxi-mini

xg = np.unique(Xmb)
yg = np.unique(Ymb)
meanR = np.full((len(yg),len(xg)),np.nan)
medianR = np.full((len(yg),len(xg)),np.nan)
varianceR = np.full((len(yg),len(xg)),np.nan)
miniR = np.full((len(yg),len(xg)),np.nan)
maxiR = np.full((len(yg),len(xg)),np.nan)
rangeR = np.full((len(yg),len(xg)),np.nan)

ix = np.searchsorted(xg,Xmb)
iy = np.searchsorted(yg,Ymb)

meanR[iy,ix] = mean.reshape([len(mean),1])
medianR[iy,ix] = median.reshape([len(median),1])
varianceR[iy,ix] = variance.reshape([len(variance),1])
miniR[iy,ix] = mini.reshape([len(mini),1])
maxiR[iy,ix] = maxi.reshape([len(maxi),1])
rangeR[iy,ix] = range.reshape([len(range),1])

data = {"x":Xmb,"y":Ymb,"mean_app_mb":mean,"median_app_mb":median,"variance":variance,"min_max_range":range}
sio.savemat(f'{savepath}exp{expNo}-apparent-mass-balance-BedMap-grid.mat',data)

In [ ]:
## PDE residual (Measures grid)

# convert from m/s to m/a
spy = 31536000.  # seconds per year
pde_res_ma = pde_res*spy

mean = np.mean(np.where(inside_v,pde_res_ma,np.nan),axis=0)
median = np.median(np.where(inside_v,pde_res_ma,np.nan),axis=0)
variance = np.var(np.where(inside_v,pde_res_ma,np.nan),axis=0)
mini = np.min(np.where(inside_v,pde_res_ma,np.nan),axis=0)
maxi = np.max(np.where(inside_v,pde_res_ma,np.nan),axis=0)
range = maxi-mini

data = {"x":Xv,"y":Yv,"mean_residual":mean,"median_residual":median,"variance":variance,"min_resiudal":mini,"max_residual":maxi}
sio.savemat(f'{savepath}exp{expNo}-pde-residual-Measures-grid.mat',data)

In [ ]:
## PDE residual (Measures grid)

# convert from m/s to m/a
spy = 31536000.  # seconds per year
pde_res_ma = pde_res*spy

res = np.where(inside_v,pde_res_ma[0,:],np.nan)

data = {"x":Xv,"y":Yv,"residual":res}
sio.savemat(f'{savepath}exp{expNo}-No1-pde-residual-Measures-grid.mat',data)

In [ ]:
## profiles
for i,profile,xy in zip(np.arange(4),[H_profile_A,H_profile_B,H_profile_C,H_profile_D],[XYA,XYB,XYC,XYD]):
    mean = np.mean(profile,axis=0)
    median = np.median(profile,axis=0)
    variance = np.var(profile,axis=0)
    mini = np.min(profile,axis=0)
    maxi = np.max(profile,axis=0)
    range = maxi-mini

    data = {"x":np.array(xy[:,0]).reshape((len(mean),1)),"y":np.array(xy[:,1]).reshape((len(mean),1)),"mean_thickness":np.array(mean).reshape((len(mean),1)),"median_thickness":np.array(median).reshape((len(mean),1)),"variance":np.array(variance).reshape((len(mean),1)),"min_max_range":np.array(range).reshape((len(mean),1))}
    sio.savemat(f'{savepath}profile-p{i+1}-exp{expNo}-ice-thickness.mat',data)